##### 5. LangChain Model 高级配置
> 在入门案例中我们通过 create_agent 创建Agent，直接指定模型名就可以了，非常方便。但模型的参数却都是默认参数，无法修改。
>
> langchain提供了两种常见方法用来初始化模型：
>
> • 使用 init_chat_model 函数，由langchain自动创建模型对象
>
> • 使用不同模型对应的Model类，手动创建模型对象
>
但最常用的还是 init_chat_model.

##### 6 模型初始化
###### 1. 原生 SDK 调用（Low-Level / Native API）
- 代码特征：from openai import OpenAI -> client.chat.completions.create(...)
- 本质：直接与各大模型厂商（或兼容 OpenAI 协议的中转服务）的底层 HTTP 接口交互。
- 特点：
  - 零框架开销：最轻量、性能最高，没有任何第三方框架的抽象损耗。
  - 完全自主可控：你可以精准控制每一个 HTTP Header、请求体参数和流式返回。
  - 缺点：如果需要切换大模型厂商（比如从 OpenAI 换成 Anthropic 或本地 Ollama），代码和数据结构往往需要重构。
```python
    """层 1：底层原生 SDK 调用"""
    # 初始化原生客户端（可配置 base_url 对接 DeepSeek、硅基流动或 OpenAI）
    client = OpenAI(
        api_key=os.getenv("DEEPSEEK_API_KEY"),
        base_url="https://api.deepseek.com/v1"
    )

    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": "一句话介绍你自己？"}],
        temperature=0.1
    )

    print("【原生 SDK 响应】:", response.choices[0].message.content)
```
###### 2. 框架层模型调用（Framework Abstraction / LLM Wrapper）
- 代码特征：我们刚才写的 ChatOpenAI、ChatOllama、ChatDeepSeek、init_chat_model。
- 本质：像 LangChain、LiteLLM 这样的框架对底层 API 进行了统一封装。
- 特点：
  - 统一接口：提供标准化的 invoke / ainvoke 和消息对象（HumanMessage、AIMessage），切换模型只需改个配置。
  - 生态丰富：自带 Prompt 模板、流式输出（Streaming）、输出解析器（Output Parsers）等辅助工具。
```python
"""层 2：框架层模型封装调用"""
    # 初始化 LangChain 统一聊天模型封装
    llm = ChatOpenAI(
        model="deepseek-chat",
        base_url="https://api.deepseek.com/v1",
        temperature=0.1
    )

    # 使用标准化消息对象调用
    response = llm.invoke([HumanMessage(content="一句话介绍你自己？")])

    print("【LangChain 封装响应】:", response.content)
```
###### 3. 智能体与工作流层（Agent & Workflow Layer）
- 代码特征：create_agent / LangGraph 的 create_react_agent、多轮工具调用循环、StateGraph。
- 本质：赋予 LLM “手”和“脑”（即 Tools 结合 Reasoning Loop），让模型不仅能生成文本，还能自主决定何时调用外部工具（如 MCP、SQL 数据库查询）、如何处理报错并进行自我纠错。
- 特点：
  - 自主决策：不再是单向的“问答”，而是“观察-思考-行动（ReAct）”的闭环。
  - 状态管理：结合图结构（LangGraph）管理复杂的会话状态、多轮记忆和人工审核机制。
```python
# 1. 定义一个工具
@tool
def get_weather(location: str) -> str:
    """查询指定城市的天气情况。"""
    return f"{location}今天天气晴朗，气温 22°C。"

async def run_agent_workflow():
    """层 3：智能体与工作流层调用"""
    # 2. 创建带有工具的 Agent
    agent = create_agent(
        model="deepseek-chat",
        tools=[get_weather],
    )

    # 3. 异步执行调用（Agent 会自主决定是否调用工具）
    response = await agent.ainvoke({
        "messages": [("user", "杭州今天天气怎么样？")]
    })

    # 打印最终回答
    print("【Agent 智能体响应】:", response["messages"][-1].content)
```
###### 方式一：统一工厂函数：init_chat_model（⭐️ 官方推荐）
> 可以用完全相同的代码风格切换不同的模型提供商（如 Ollama、OpenAI、Anthropic 等），只需在字符串中指定供应商前缀或通过参数控制。
>
> ```python
> from langchain.chat_models import init_chat_model
>
> # 1. 初始化 Ollama 本地模型
> llm_ollama = init_chat_model("ollama:deepseek-r1:14b")
>
> # 2. 初始化 OpenAI 模型
> llm_openai = init_chat_model("gpt-4o", model_provider="openai")
>
> # 调用示例
> response = llm_ollama.invoke("你好")
> print(response.content)
> ```
###### 方式二：厂商专用类初始化（传统显式模式）
> 直接导入并使用特定厂商封装好的 Chat 类（如 ChatOllama、ChatOpenAI、ChatAnthropic 等）。这种方式最直观，且能够直接访问该厂商特有的底层配置参数。
> ```python
> # 需要先安装对应依赖: uv add langchain-ollama
> from langchain_ollama import ChatOllama
>
> # 显式使用 ChatOllama 类
> llm = ChatOllama(
>     model="deepseek-r1:14b",
>     temperature=0.7,
>     # 可传入其他专属参数
> )
>
> response = llm.invoke("介绍一下你自己")
> print(response.content)
> ```
###### 方式三：OpenAI 兼容模式初始化（适用于第三方 API / DeepSeek / 本地 vLLM）
> 如果你的模型提供商（例如 DeepSeek 官方 API、SiliconFlow、Moonshot 或本地部署的 vLLM）兼容 OpenAI 的接口格式，你可以直接使用 ChatOpenAI 并通过 base_url 指向自定义的端点。
> ```python
> # 需要先安装: uv add langchain-openai
> from langchain_openai import ChatOpenAI
>
> # 接入 DeepSeek 官方 API 或兼容 OpenAI 格式的第三方服务
> llm = ChatOpenAI(
>     model="deepseek-chat",
>     base_url="https://api.deepseek.com/v1",  # 替换为你的目标 API 地址
>     api_key="your-api-key-here",            # 或通过环境变量 DEEPSEEK_API_KEY 自动读取
>     temperature=0.1
> )
>
> response = llm.invoke("你好，DeepSeek！")
> print(response.content)
> ```

In [1]:
# ===================1.初始化模型=====================
# 导入Langchain的初始化模型的函数
# 1. 加载环境变量
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

# 2.调用init_chat_model初始化模型，LangChain自动推断base_url和api_key，我们配置其它参数即可
model = init_chat_model(
    model="deepseek-v4-flash", temperature=0.7, extra_body={"thinking": {"type": "disabled"}}
)

# =================2.测试====================
# 3.初始化Agent，传入初始化好的model、tools，tools可以为空
agent = create_agent(model=model, tools=[])

# 4.调用Agent
response = agent.invoke({"messages": [{"role": "user", "content": "你是谁？"}]})
print(response)

{'messages': [HumanMessage(content='你是谁？', additional_kwargs={}, response_metadata={}, id='818924e1-b663-43e8-94d0-fe6519cbf96f'), AIMessage(content='你好！我是DeepSeek，由深度求索公司创造的AI助手。😊\n\n我可以帮你解答各种问题、进行对话、提供信息、协助写作、分析问题等等。我支持文件上传（图像、txt、pdf、ppt、word、excel），可以从中读取文字信息，也能直接查看图片内容。我还有1M的上下文长度，能一次性处理像《三体》三部曲那样体量的书籍！\n\n而且重点是——我完全免费！App端还支持语音输入，你可以通过官方应用商店下载使用。\n\n有什么我可以帮你的吗？尽管问吧！✨', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 123, 'prompt_tokens': 6, 'total_tokens': 129, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 6}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': 'b08c725c-de2e-4e25-8ce2-e30e39d0e64b', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1e804-421d-75d1-9d9c-